# RoadAnomaly — Stage 5 final test evaluation

**Use this notebook only.**

1. Open this file in Colab (link from GitHub)
2. **Runtime → Change runtime type → GPU (T4)** → Save
3. **Runtime → Run all**

It mounts Drive, pulls latest code, loads E0/E1/E2a/E2b with a Keras-safe loader, and evaluates **once** on the locked test split.

Outputs on Drive:
- `RoadAnomaly/experiments/results/phase4/phase4_final_test_comparison.csv`
- `RoadAnomaly/experiments/metadata/phase4/STAGE5_TEST_EVAL_DONE.json`


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, tempfile, zipfile
import pandas as pd

DRIVE_ROOT = Path("/content/drive/MyDrive/RoadAnomaly")
CLONE_DIR = Path("/content/RoadAnomaly")
GITHUB_REPO = "https://github.com/AhmedChatti/RoadAnomaly.git"
GITHUB_BRANCH = "main"
GITHUB_TOKEN_SECRET_NAME = "GITHUB_TOKEN"
EXPECTED_SPLIT = "9a01c05cfc7d33783c4f64b91b0ae423a4d3ad57439fd71ffc029a8f04585c78"
AUTHORIZE_TEST_EVALUATION = True
print("config ok")


In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")

os.environ["ROADANOMALY_ARTIFACTS_ROOT"] = str(DRIVE_ROOT)
os.environ["ROADANOMALY_FRAMES_DIR"] = str(DRIVE_ROOT / "dataset" / "original" / "frames")
os.environ["ROADANOMALY_CHECKPOINTS_DIR"] = str(DRIVE_ROOT / "checkpoints")
os.environ["ROADANOMALY_METADATA_DIR"] = str(DRIVE_ROOT / "experiments" / "metadata")
os.environ["ROADANOMALY_RESULTS_DIR"] = str(DRIVE_ROOT / "experiments" / "results")
os.environ["ROADANOMALY_FIGURES_DIR"] = str(DRIVE_ROOT / "experiments" / "figures")

token = userdata.get(GITHUB_TOKEN_SECRET_NAME)
auth_url = GITHUB_REPO.replace("https://", f"https://x-access-token:{token}@")

def run(cmd, cwd=None):
    print("+", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if not CLONE_DIR.exists():
    run(["git", "clone", "--branch", GITHUB_BRANCH, auth_url, str(CLONE_DIR)])
else:
    run(["git", "remote", "set-url", "origin", auth_url], cwd=str(CLONE_DIR))
    run(["git", "fetch", "origin"], cwd=str(CLONE_DIR))
    run(["git", "checkout", "-B", GITHUB_BRANCH, f"origin/{GITHUB_BRANCH}"], cwd=str(CLONE_DIR))
    run(["git", "reset", "--hard", f"origin/{GITHUB_BRANCH}"], cwd=str(CLONE_DIR))

print("HEAD", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(CLONE_DIR), text=True).strip())
os.environ["ROADANOMALY_ROOT"] = str(CLONE_DIR)
sys.path.insert(0, str(CLONE_DIR / "src"))
os.chdir(CLONE_DIR)
print("cwd", Path.cwd())


In [ ]:
assert AUTHORIZE_TEST_EVALUATION, "Flip AUTHORIZE_TEST_EVALUATION only for the approved one-shot run."

from dataset import load_split_manifest, verify_split_lock
from evaluation import evaluate_predictor_on_split
from models.baseline import AllBackgroundModel
from models.cnn import build_scratch_fcn
from models.unet_mobilenetv2 import build_unet_mobilenetv2
from losses import bce_dice_loss
from tensorflow import keras

_INIT_NAMES = {
    "GlorotUniform", "GlorotNormal", "HeNormal", "HeUniform", "LecunNormal",
    "LecunUniform", "VarianceScaling", "RandomNormal", "RandomUniform",
    "TruncatedNormal", "Orthogonal",
}

def _strip_axes(node):
    if isinstance(node, dict):
        if node.get("class_name") in _INIT_NAMES and isinstance(node.get("config"), dict):
            node["config"].pop("input_axes", None)
            node["config"].pop("output_axes", None)
        for v in node.values():
            _strip_axes(v)
    elif isinstance(node, list):
        for item in node:
            _strip_axes(item)

def load_weights_from_keras_zip(model, path):
    path = Path(path)
    with tempfile.TemporaryDirectory() as tmp:
        tmp_path = Path(tmp)
        with zipfile.ZipFile(path, "r") as zin:
            names = {n.replace("\\", "/"): n for n in zin.namelist()}
            key = next((k for k in names if k.endswith("model.weights.h5")), None)
            if key is None:
                raise FileNotFoundError(f"No model.weights.h5 in {path}")
            out = tmp_path / "model.weights.h5"
            out.write_bytes(zin.read(names[key]))
        model.load_weights(out)

def load_checkpoint(eid: str, ckpt: Path):
    # Rebuild architecture + load weights (avoids Colab/local Keras mismatch).
    if eid.startswith("E1"):
        model = build_scratch_fcn(288, 512, base_filters=8, dropout=0.2)
    elif eid.startswith("E2"):
        model = build_unet_mobilenetv2(
            input_height=288, input_width=512, weights=None, encoder_trainable=False
        )
    else:
        raise ValueError(eid)
    try:
        load_weights_from_keras_zip(model, ckpt)
        print("  loaded via rebuild+weights")
        return model
    except Exception as exc:
        print("  rebuild+weights failed:", type(exc).__name__, exc)
        print("  trying sanitized load_model")
        with tempfile.TemporaryDirectory() as tmp:
            sanitized = Path(tmp) / "sanitized.keras"
            with zipfile.ZipFile(ckpt, "r") as zin, zipfile.ZipFile(sanitized, "w") as zout:
                for info in zin.infolist():
                    data = zin.read(info.filename)
                    name = info.filename.replace("\\", "/")
                    if name.endswith("config.json"):
                        cfg = json.loads(data.decode("utf-8"))
                        _strip_axes(cfg)
                        data = json.dumps(cfg).encode("utf-8")
                    zout.writestr(info, data)
            return keras.models.load_model(
                sanitized,
                custom_objects={"bce_dice": bce_dice_loss(positive_weight=15.0)},
                compile=False,
            )

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

split_sha = verify_split_lock(load_split_manifest())
assert split_sha == EXPECTED_SPLIT
print("split_ok", split_sha)

FREEZE = json.loads('{"frozen_at_utc":"2026-11-08T17:45:00+00:00","purpose":"Freeze validation-based model and threshold choices before any test-set access (Stage 5).","split_manifest_sha256":"9a01c05cfc7d33783c4f64b91b0ae423a4d3ad57439fd71ffc029a8f04585c78","git_commit_sha_training":"9e734a1d0af48342f92c084216b1dd1d4c254741","selection_split":"val","selection_metric":"mean_per_image_mean_dice","primary_model":{"experiment_id":"E2b_unet_mobilenetv2_finetune_20261108_1721","reason":"Highest validation mean Dice among E0/E1/E2a/E2b","operational_threshold":0.55,"training_threshold":0.5,"checkpoint_path_drive":"/content/drive/MyDrive/RoadAnomaly/checkpoints/phase4/E2b_unet_mobilenetv2_finetune_20261108_1721/best.keras","checkpoint_sha256":"280170dc64347b0b4cc8ce3af9be40dec4ac3733b27d1abeb620cc0c8e88e432"},"models_to_evaluate_once_on_test":[{"experiment_id":"E0_all_background","operational_threshold":0.5,"normalize":"scale_0_1","notes":"trivial baseline; no checkpoint"},{"experiment_id":"E1_cnn_scratch_20261108_1510","operational_threshold":0.35,"normalize":"scale_0_1","checkpoint_path_drive":"/content/drive/MyDrive/RoadAnomaly/checkpoints/phase3/E1_cnn_scratch_20261108_1510/best.keras","checkpoint_sha256":"567bde326ee5b776bc34132b23012543b35f195ec8d60229d01196eb1a0fb234"},{"experiment_id":"E2a_unet_mobilenetv2_frozen_20261108_1714","operational_threshold":0.15,"normalize":"mobilenet_v2","checkpoint_path_drive":"/content/drive/MyDrive/RoadAnomaly/checkpoints/phase4/E2a_unet_mobilenetv2_frozen_20261108_1714/best.keras","checkpoint_sha256":"428845e976a8f4773ce929ec66520227b49928142d4c34e38974af671f38b080"},{"experiment_id":"E2b_unet_mobilenetv2_finetune_20261108_1721","operational_threshold":0.55,"normalize":"mobilenet_v2","checkpoint_path_drive":"/content/drive/MyDrive/RoadAnomaly/checkpoints/phase4/E2b_unet_mobilenetv2_finetune_20261108_1721/best.keras","checkpoint_sha256":"280170dc64347b0b4cc8ce3af9be40dec4ac3733b27d1abeb620cc0c8e88e432"}],"rules":["Do not retune thresholds on test.","Do not change model selection using test metrics.","Evaluate each listed model at most once on test under this freeze.","allow_test=True only inside the Stage 5 script/notebook."]}')
freeze_path = DRIVE_ROOT / "experiments/metadata/phase4/FINAL_MODEL_SELECTION_FREEZE.json"
freeze_path.parent.mkdir(parents=True, exist_ok=True)
freeze_path.write_text(json.dumps(FREEZE, indent=2), encoding="utf-8")
freeze = FREEZE
print("primary", freeze["primary_model"]["experiment_id"])
print("Wrote freeze", freeze_path)

out_dir = DRIVE_ROOT / "experiments/results/phase4"
meta_dir = DRIVE_ROOT / "experiments/metadata/phase4"
out_dir.mkdir(parents=True, exist_ok=True)
meta_dir.mkdir(parents=True, exist_ok=True)

guard = meta_dir / "STAGE5_TEST_EVAL_DONE.json"
if guard.exists():
    raise RuntimeError(
        f"Test evaluation already recorded at {guard}. "
        "Delete that file on Drive only if you intentionally need a re-run."
    )

rows = []
for spec in freeze["models_to_evaluate_once_on_test"]:
    eid = spec["experiment_id"]
    thr = float(spec["operational_threshold"])
    normalize = spec["normalize"]
    print("Evaluating", eid, "thr", thr)

    if eid.startswith("E0"):
        model = AllBackgroundModel()
        predict_fn = lambda x, m=model: m.predict_proba(x)
        ckpt_sha = None
        ckpt_ok = True
    else:
        ckpt = Path(spec["checkpoint_path_drive"])
        assert ckpt.is_file(), f"Missing checkpoint: {ckpt}"
        ckpt_sha = sha256_file(ckpt)
        expected = spec["checkpoint_sha256"]
        ckpt_ok = ckpt_sha == expected
        print("  sha", ckpt_sha, "match", ckpt_ok)
        assert ckpt_ok, (ckpt_sha, expected)
        loaded = load_checkpoint(eid, ckpt)
        predict_fn = lambda x, m=loaded: m.predict(x, verbose=0)

    result = evaluate_predictor_on_split(
        predict_fn,
        "test",
        threshold=thr,
        normalize=normalize,
        batch_size=4,
        allow_test=True,
    )
    per = result["per_image"]
    per_path = out_dir / f"{eid}_test_per_image.csv"
    per.to_csv(per_path, index=False)
    mp = result["mean_per_image"]
    rows.append({
        "experiment_id": eid,
        "split": "test",
        "threshold": thr,
        "normalize": normalize,
        "checkpoint_sha256": ckpt_sha,
        "checkpoint_sha_match": ckpt_ok,
        **mp,
        "global_dice": result["global"]["dice"],
        "global_iou": result["global"]["iou"],
        "global_accuracy": result["global"]["accuracy"],
        "per_image_csv": str(per_path),
    })
    print("  mean_dice", mp.get("mean_dice"))

summary = pd.DataFrame(rows)
summary_path = out_dir / "phase4_final_test_comparison.csv"
summary.to_csv(summary_path, index=False)
report = {
    "stage": 5,
    "status": "completed",
    "split": "test",
    "split_manifest_sha256": split_sha,
    "freeze_file": str(freeze_path),
    "primary_model": freeze["primary_model"]["experiment_id"],
    "summary_csv": str(summary_path),
    "rows": rows,
    "notes": "One-shot test evaluation after frozen validation selection. No threshold retuning. Keras-safe rebuild+weights loader.",
}
guard.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(summary)
print("DONE")
print("Wrote", summary_path)
print("Wrote", guard)
